In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class CRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # keeping the layers we want from the CNN
        self.cnn = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        # first iteration we're freezing CNN and only training RNN and CTC head
        for param in self.cnn.parameters():
            param.requires_grad = False

        # use this to force height of images down to be 1
        self.height_pool = nn.AdaptiveAvgPool2d((1, None))

        # use Bi-LSTM as the RNN
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

        # input to linear layer is twice the hidden size since using bidirectional LSTM
        self.fc = nn.Linear(256*2, vocab_size)

    def forward(self, x):
        features = self.cnn(x)
        # collapse everything down to height of 1
        # allows use of images from different datasets without worrying about different heights
        features = self.height_pool(features)
        # remove the height dimension to go to 2D
        features = features.squeeze(2)
        # reorder the tensors so they're in format to be fed into RNN
        # from documentation it needs to be in format of (N,L,Hin) and output of cnn was (B, C, H, W)
        # after removing height dimension features is of format (B, C, W)
            # N is batch size (B)
            # L is sequence length (W)
            # Hin is input length (C)
        features = features.permute(0,2,1)  # (B,W,C)

        seq, _ = self.rnn(features)
        out = self.fc(seq)

        return out